In [1]:

import os
import numpy as np
import pandas as pd
from lightfm import LightFM
from lightfm.cross_validation import random_train_test_split
from lightfm.evaluation import precision_at_k
from lightfm.data import Dataset
import optuna


/home/egor/miniconda3/envs/lightfm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
RESULT_PREPROCESSED_PATH = r'/mnt/c/Users/Egor/Desktop/study/diploma/project/dataset_preprocessed'


In [3]:
def reduce_mem_usage(df):
    numerics = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
    start_mem = df.memory_usage().sum() / 1024 ** 2
    for col in df.columns:
        col_type = df[col].dtypes
        if col_type in numerics:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                else:
                    df[col] = df[col].astype(np.int64)
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
    end_mem = df.memory_usage().sum() / 1024 ** 2
    print(f'Memory: {start_mem:.2f} Mb -> {end_mem:.2f} Mb ({100 * (start_mem - end_mem) / start_mem:.1f}% reduction)')
    return df


def load_data(data_dir):
    print("Загрузка данных...")
    transactions = reduce_mem_usage(pd.read_csv(os.path.join(data_dir, "transactions.csv")))
    articles = reduce_mem_usage(pd.read_csv(os.path.join(data_dir, "articles.csv")))
    customers = reduce_mem_usage(pd.read_csv(os.path.join(data_dir, "customers.csv")))
    rfm_features = reduce_mem_usage(pd.read_csv(os.path.join(data_dir, "rfm_features.csv")))
    return transactions, articles, customers, rfm_features


In [4]:
transactions, articles, customers, rfm = load_data(RESULT_PREPROCESSED_PATH)
customers = customers.merge(rfm, on='customer_id', how='left')


Загрузка данных...
Memory: 2197.69 Mb -> 796.66 Mb (63.7% reduction)
Memory: 21.74 Mb -> 13.99 Mb (35.6% reduction)
Memory: 83.74 Mb -> 56.26 Mb (32.8% reduction)
Memory: 83.15 Mb -> 36.38 Mb (56.2% reduction)


In [5]:

user_cat_cols = ['age_group', 'club_member_status', 'fashion_news_frequency']
# Бинарные фичи в строковом представлении
customers['Active'] = customers['Active'].fillna(0).astype(int).astype(str).apply(lambda x: f"Active_{x}")
customers['FN'] = customers['FN'].fillna(0).astype(int).astype(str).apply(lambda x: f"FN_{x}")
user_cat_cols += ['Active', 'FN']

# Добавляем RFM-метрики, предварительно превратив их в категории (бинаризация)
# Для recency, frequency, monetary, avg_basket, n_unique_articles
# weekend_purchase_ratio уже в [0,1] – можно разбить на 5 категорий
# fav_product_group – уже категория

rfm_cat_cols = []

# recency (дней с последней покупки) – разбиваем на 5 квантилей
customers['recency_cat'] = pd.qcut(customers['recency'].fillna(customers['recency'].median()), 
                                   q=5, labels=['rec_1','rec_2','rec_3','rec_4','rec_5'])
rfm_cat_cols.append('recency_cat')

# frequency (число покупок) – разбиваем на 5 квантилей
customers['frequency_cat'] = pd.qcut(customers['frequency'].fillna(0), 
                                     q=5, labels=['freq_1','freq_2','freq_3','freq_4','freq_5'])
rfm_cat_cols.append('frequency_cat')

# n_unique_articles – квантили
customers['unique_articles_cat'] = pd.qcut(customers['n_unique_articles'].fillna(0), 
                                           q=5, labels=['uniq_1','uniq_2','uniq_3','uniq_4','uniq_5'])
rfm_cat_cols.append('unique_articles_cat')

# fav_product_group – уже категория, заполняем пропуски
customers['fav_product_group'] = customers['fav_product_group'].fillna('Unknown')
rfm_cat_cols.append('fav_product_group')

# Добавляем все RFM-категории в общий список фичей пользователя
user_cat_cols += rfm_cat_cols




In [6]:

# ===============================
# 3. ФИЧИ ДЛЯ ТОВАРОВ (как ранее)
# ===============================
item_cat_cols = ['product_group_name', 'colour_group_name']

# ===============================
# 4. ПОСТРОЕНИЕ DATASET ДЛЯ LIGHTFM
# ===============================
all_user_ids = transactions['customer_id'].unique()
all_article_ids = transactions['article_id'].unique()
customers = customers[customers['customer_id'].isin(all_user_ids)].copy()
articles = articles[articles['article_id'].isin(all_article_ids)].copy()

# Все возможные значения фичей пользователя
user_feature_names = []
for col in user_cat_cols:
    user_feature_names.extend([f"{col}_{v}" for v in customers[col].unique()])

# Все возможные значения фичей товаров
item_feature_names = []
for col in item_cat_cols:
    item_feature_names.extend([f"{col}_{v}" for v in articles[col].unique()])


In [7]:
print(user_feature_names)
print(item_feature_names)

['age_group_40-49', 'age_group_20-29', 'age_group_50-59', 'age_group_30-39', 'age_group_<20', 'age_group_60+', 'club_member_status_ACTIVE', 'club_member_status_UNKNOWN', 'club_member_status_PRE-CREATE', 'club_member_status_LEFT CLUB', 'fashion_news_frequency_NONE', 'fashion_news_frequency_Regularly', 'fashion_news_frequency_Monthly', 'Active_Active_0', 'Active_Active_1', 'FN_FN_0', 'FN_FN_1', 'recency_cat_rec_1', 'recency_cat_rec_2', 'recency_cat_rec_5', 'recency_cat_rec_4', 'recency_cat_rec_3', 'frequency_cat_freq_4', 'frequency_cat_freq_5', 'frequency_cat_freq_1', 'frequency_cat_freq_3', 'frequency_cat_freq_2', 'unique_articles_cat_uniq_4', 'unique_articles_cat_uniq_5', 'unique_articles_cat_uniq_1', 'unique_articles_cat_uniq_3', 'unique_articles_cat_uniq_2', 'fav_product_group_Garment Upper body', 'fav_product_group_Underwear', 'fav_product_group_Garment Lower body', 'fav_product_group_Garment Full body', 'fav_product_group_Swimwear', 'fav_product_group_Accessories', 'fav_product_gro

In [8]:

dataset = Dataset()
dataset.fit(users=all_user_ids,
            items=all_article_ids,
            user_features=user_feature_names,
            item_features=item_feature_names)

def iter_interactions(df):
    for row in df.itertuples():
        yield row.customer_id, row.article_id

interactions_matrix, _ = dataset.build_interactions(iter_interactions(transactions))

def build_user_features(dataset, customers_df, cat_cols):
    features = []
    for _, row in customers_df.iterrows():
        uid = row['customer_id']
        feat_vals = [f"{col}_{row[col]}" for col in cat_cols if pd.notna(row[col])]
        features.append((uid, feat_vals))
    return dataset.build_user_features(features)

# Матрица признаков товаров
def build_item_features(dataset, articles_df, cat_cols):
    features = []
    for _, row in articles_df.iterrows():
        aid = row['article_id']
        feat_vals = [f"{col}_{row[col]}" for col in cat_cols if pd.notna(row[col])]
        features.append((aid, feat_vals))
    return dataset.build_item_features(features)


In [9]:

user_feature_matrix = build_user_features(dataset, customers, user_cat_cols)

In [10]:
item_feature_matrix = build_item_features(dataset, articles, item_cat_cols)

print(f"interactions shape: {interactions_matrix.shape}")
print(f"user features shape: {user_feature_matrix.shape}")
print(f"item features shape: {item_feature_matrix.shape}")

interactions shape: (1362281, 104547)
user features shape: (1362281, 1362331)
item features shape: (104547, 104616)


In [11]:
train_interactions, test_interactions  = random_train_test_split(interactions_matrix, test_percentage=0.2, random_state=42)



In [12]:

N_SAMPLE_USERS = 5000
np.random.seed(42)
test_interactions = test_interactions.tocsr()

sampled_test_users = np.random.choice(test_interactions.shape[0], size=min(N_SAMPLE_USERS, test_interactions.shape[0]), replace=False)
test_sampled = test_interactions[sampled_test_users]


In [13]:

def objective(trial):
    learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-3)
    no_components = trial.suggest_int('no_components', 30, 80)
    loss = trial.suggest_categorical('loss', ['warp', 'bpr'])
    item_alpha = trial.suggest_loguniform('item_alpha', 1e-4, 1e-1)
    user_alpha = trial.suggest_loguniform('user_alpha', 1e-4, 1e-1)
    max_sampled = trial.suggest_int('max_sampled', 10, 50)
    
    model = LightFM(no_components=no_components,
                    learning_rate=learning_rate,
                    loss=loss,
                    item_alpha=item_alpha,
                    user_alpha=user_alpha,
                    max_sampled=max_sampled,
                    random_state=42)
    
    model.fit(train_interactions,
              user_features=user_feature_matrix,
              item_features=item_feature_matrix,
              epochs=15,
              num_threads=20,
              verbose=True)
    
    prec = precision_at_k(model, test_sampled,
                          user_features=user_feature_matrix,
                          item_features=item_feature_matrix,
                          k=12,
                          num_threads=20).mean()
    return prec


study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=20, show_progress_bar=True)

print("Лучшие гиперпараметры:", study.best_params)
print("Лучшее Precision@12:", study.best_value)

[I 2026-04-25 23:47:37,173] A new study created in memory with name: no-name-4e443a2c-e474-4f7c-8406-8f40f16eca53
  0%|                                                                                            | 0/20 [00:00<?, ?it/s]/tmp/ipykernel_765/1307238766.py:2: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-3)
/tmp/ipykernel_765/1307238766.py:5: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  item_alpha = trial.suggest_loguniform('item_alpha', 1e-4, 1e-1)
/tmp/ipykernel_765/1307238766.py:6: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See h

[I 2026-04-26 00:02:40,694] Trial 0 finished with value: 0.001863828394562006 and parameters: {'learning_rate': 5.6115164153345e-05, 'no_components': 78, 'loss': 'warp', 'item_alpha': 0.00029380279387035364, 'user_alpha': 0.00029375384576328325, 'max_sampled': 12}. Best is trial 0 with value: 0.001863828394562006.



Best trial: 0. Best value: 0.00186383:  10%|████▏                                     | 2/20 [28:29<4:13:47, 845.96s/it]

[I 2026-04-26 00:16:06,366] Trial 1 finished with value: 0.0017964610597118735 and parameters: {'learning_rate': 0.0005399484409787432, 'no_components': 60, 'loss': 'warp', 'item_alpha': 0.0812324508558869, 'user_alpha': 0.03142880890840111, 'max_sampled': 18}. Best is trial 0 with value: 0.001863828394562006.



Best trial: 0. Best value: 0.00186383:  15%|██████▎                                   | 3/20 [42:29<3:58:54, 843.23s/it]

...

[I 2026-04-26 03:51:23,856] Trial 17 finished with value: 0.0017964610597118735 and parameters: {'learning_rate': 8.316264993185367e-05, 'no_components': 73, 'loss': 'warp', 'item_alpha': 0.013226338130332287, 'user_alpha': 0.0001162719671524867, 'max_sampled': 14}. Best is trial 0 with value: 0.001863828394562006.



Best trial: 0. Best value: 0.00186383:  95%|██████████████████████████████████████▉  | 19/20 [4:17:52<13:59, 839.57s/it]

[I 2026-04-26 04:05:29,962] Trial 18 finished with value: 0.0017964610597118735 and parameters: {'learning_rate': 0.0007501054462493084, 'no_components': 63, 'loss': 'warp', 'item_alpha': 0.0036492025646553835, 'user_alpha': 0.0018859099610952859, 'max_sampled': 25}. Best is trial 0 with value: 0.001863828394562006.



Best trial: 0. Best value: 0.00186383: 100%|█████████████████████████████████████████| 20/20 [4:30:51<00:00, 812.57s/it]

[I 2026-04-26 04:18:28,480] Trial 19 finished with value: 0.0017964610597118735 and parameters: {'learning_rate': 0.00032097864674854933, 'no_components': 53, 'loss': 'warp', 'item_alpha': 0.00027129420896956897, 'user_alpha': 0.03323621618577025, 'max_sampled': 14}. Best is trial 0 with value: 0.001863828394562006.
Лучшие гиперпараметры: {'learning_rate': 5.6115164153345e-05, 'no_components': 78, 'loss': 'warp', 'item_alpha': 0.00029380279387035364, 'user_alpha': 0.00029375384576328325, 'max_sampled': 12}
Лучшее Precision@12: 0.001863828394562006


In [17]:
best = study.best_params
final_model = LightFM(no_components=best['no_components'],
                      learning_rate=best['learning_rate'],
                      loss=best['loss'],
                      item_alpha=best['item_alpha'],
                      user_alpha=best['user_alpha'],
                      max_sampled=best.get('max_sampled', 20),
                      random_state=42)
final_model.fit(train_interactions,
                user_features=user_feature_matrix,
                item_features=item_feature_matrix,
                epochs=50,
                num_threads=4,
                verbose=True)

final_prec = precision_at_k(final_model, test_sampled,
                            user_features=user_feature_matrix,
                            item_features=item_feature_matrix,
                            k=12,
                            num_threads=4).mean()
print(f"Финальная Precision@12 на подвыборке: {final_prec:.4f}")

Epoch: 100%|████████████████████████████████████████████████████████████████████████████| 50/50 [39:09<00:00, 47.00s/it]


Финальная Precision@12 на подвыборке: 0.0052


In [18]:
import pickle

with open("lightfm_model.pkl", "wb") as f:
    pickle.dump(final_model, f)